# Stage 2: Metadata Generation for Code Examples

In this stage, we'll learn how to:
1. Process code example PDFs into chunks
2. Generate LLM-based summaries (2-3 lines) for each code snippet
3. Extract domain-agnostic keywords (5-7 per chunk)
4. Bridge the vocabulary gap between natural language queries and code syntax

Key Concept: Code syntax (e.g., "model.addVar", "GRB.MINIMIZE") doesn't match
user queries (e.g., "minimize cost", "binary variables"). We generate natural
language metadata to enable semantic search over code examples.

Output: tutorial_chunks_code.pkl (code chunks with metadata)

In [1]:
import fitz
import re
import os
import pickle
from tqdm import tqdm

from config import (
    PDF_CODE_EXAMPLES,
    CODE_CHUNKS_PATH,
    MAX_CHUNK_WORDS,
    NUM_EXAMPLE_CODE_METADATA,
    VERBOSE,
    OPENROUTER_MODEL
)
from llm_helper import get_llm

## SECTION 0: HELPER FUNCTIONS FROM PREVIOUS STAGE

In [2]:
def tokenize(text: str) -> list:
    """
    Naive word splitting for approximate token counting.

    Args:
        text: Input text string

    Returns:
        List of words (tokens)

    Note: This is a simple whitespace-based tokenizer. For production,
    consider using tiktoken for accurate GPT token counts.
    """
    return text.split()


def split_by_token_limit(text: str, max_words: int) -> list:
    """
    Splits a large string into sub-chunks if it exceeds max_words.

    This ensures that even if a single section is very long (e.g., 1000 words),
    we break it into smaller pieces that fit within the LLM context window.

    Args:
        text: Input text to split
        max_words: Maximum words per chunk

    Returns:
        List of text chunks, each <= max_words

    Example:
        >>> text = "word " * 1000  # 1000 words
        >>> chunks = split_by_token_limit(text, 400)
        >>> len(chunks)  # Will be 3 chunks: [400, 400, 200]
        3
    """
    words = tokenize(text)
    chunks = []
    current = []

    for w in words:
        if len(current) + 1 > max_words:
            chunks.append(" ".join(current))
            current = []
        current.append(w)

    if current:
        chunks.append(" ".join(current))

    return chunks

def smart_heading_match(heading_text: str, line_text: str) -> bool:
    """
    Returns True if line_text strongly matches heading_text at the start,
    with an optional numeric prefix.

    This handles various heading formats in technical documents:
    - "1.1 Variables" matches heading "Variables"
    - "1.2.3.4 Continuous Variables" matches heading "Continuous Variables"
    - "Variables." matches heading "Variables" (ignores trailing punctuation)

    Args:
        heading_text: Expected heading from PDF outline (e.g., "Variables")
        line_text: Actual line text from PDF (e.g., "1.1 Variables")

    Returns:
        True if match found, False otherwise

    Why This Matters:
        PDFs often have numbered sections (1.1, 1.2.3) but the outline may only
        contain the heading text without numbers. This function bridges that gap.
    """
    heading = heading_text.strip().lower()
    line = line_text.strip().lower()

    # Remove trailing punctuation from line
    line_no_punct = re.sub(r'[.:;\s]+$', '', line)

    # Pattern: optional numeric prefix (e.g., 1.2.3) + optional whitespace + heading text
    pattern = re.compile(
        r'^'                        # Start of line
        r'(?:\d+(?:\.\d+)*)?'       # Optional 1.1.1 style prefix
        r'\s*'                      # Optional spaces
        + re.escape(heading) +      # The heading text itself
        r'(\b|$)',                  # Word boundary or end of line
        re.IGNORECASE
    )

    return bool(pattern.match(line_no_punct))
    
def read_pdf_lines(pdf_path: str) -> dict:
    """
    Extract all text lines from PDF, organized by page.

    Args:
        pdf_path: Path to PDF file

    Returns:
        Dictionary mapping page_idx -> list of (line_idx, line_text) tuples

    Example Output:
        {
            0: [(0, "Chapter 1: Introduction"), (1, "This document..."), ...],
            1: [(0, "1.1 Variables"), (1, "Variables are..."), ...],
            ...
        }
    """
    doc = fitz.open(pdf_path)
    page_lines = {}

    for p in range(len(doc)):
        text = doc[p].get_text()
        if not text.strip():
            continue  # Skip empty pages

        raw_lines = text.split("\n")
        cleaned = []

        for i, ln in enumerate(raw_lines):
            ln_stripped = ln.strip()
            if ln_stripped:
                cleaned.append((i, ln_stripped))

        if cleaned:
            cleaned.sort(key=lambda x: x[0])
            page_lines[p] = cleaned

    doc.close()
    return page_lines


def read_outline_headings(pdf_path: str) -> list:
    """
    Extract table of contents (outline) from PDF.

    The outline provides the hierarchical structure of the document:
    - Level 1: Chapters (e.g., "Variables")
    - Level 2: Sections (e.g., "Continuous Variables")
    - Level 3: Subsections (e.g., "Variable Bounds")

    Args:
        pdf_path: Path to PDF file

    Returns:
        List of heading dictionaries with keys:
        - 'title': Heading text
        - 'level': Hierarchy level (1, 2, 3, ...)
        - 'page_idx': Page number (0-indexed)
        - 'line_idx': Will be set to None initially

    Why This Matters:
        The outline tells us WHERE sections begin. We'll later find the exact
        line number by matching heading text with page content.
    """
    doc = fitz.open(pdf_path)
    raw_toc = doc.get_toc(simple=True)
    doc.close()

    if not raw_toc:
        print("Warning: No outline found in PDF")
        return []

    headings = []
    for item in raw_toc:
        if len(item) < 3:
            continue

        lvl, ttl, pnum = item[:3]
        headings.append({
            'title': ttl,
            'level': lvl,
            'page_idx': pnum - 1,  # Convert to 0-indexed
            'line_idx': None        # Will be populated later
        })

    # Sort by page, then by level (in case multiple headings on same page)
    headings.sort(key=lambda h: (h['page_idx'], h['level']))
    return headings

def locate_heading_lines(page_lines: dict, headings: list):
    """
    Find the exact line number where each heading appears in the PDF text.

    This function modifies the headings list in-place, setting the 'line_idx'
    field for each heading.

    Args:
        page_lines: Output from read_pdf_lines()
        headings: Output from read_outline_headings()

    Strategy:
        For each heading:
        1. Look at its assigned page
        2. Search for a line that matches the heading text (using smart_heading_match)
        3. If found, record that line index
        4. If not found, place heading after the last line (fallback)

    Why This Matters:
        We need precise line positions to know where each section starts and ends.
        Without this, we can't extract clean section text.
    """
    for h in headings:
        p = h['page_idx']

        if p not in page_lines:
            h['line_idx'] = 0
            continue

        lines_on_page = page_lines[p]
        found_line = None

        # Search for matching line
        for (li, txt) in lines_on_page:
            if smart_heading_match(h['title'], txt):
                found_line = li
                break

        if found_line is not None:
            h['line_idx'] = found_line
        else:
            # Fallback: place after last line
            last_li = lines_on_page[-1][0]
            h['line_idx'] = last_li + 1

    # Re-sort by (page, line) to ensure proper ordering
    headings.sort(key=lambda x: (x['page_idx'], x['line_idx']))

def gather_lines_between(page_lines: dict, start_h: dict, end_h: dict) -> list:
    """
    Collect all text lines from start_h to end_h (exclusive).

    This handles three cases:
    1. Both headings on same page: Collect lines between them
    2. Headings on different pages: Collect from start page to end page
    3. Edge cases: Empty pages, missing content

    Args:
        page_lines: Page text dictionary
        start_h: Starting heading (inclusive)
        end_h: Ending heading (exclusive)

    Returns:
        List of text lines between the two headings
    """
    collected = []
    sp, sl = start_h['page_idx'], start_h['line_idx']
    ep, el = end_h['page_idx'], end_h['line_idx']

    page = sp
    while True:
        if page not in page_lines:
            if page == ep:
                break
            page += 1
            continue

        lines_this_page = page_lines[page]

        if page == sp and page == ep:
            # Case 1: Both headings on same page
            for (li, txt) in lines_this_page:
                if li >= sl and li < el:
                    collected.append(txt)
            break
        elif page == sp:
            # Case 2a: Starting page (collect from sl to end)
            for (li, txt) in lines_this_page:
                if li >= sl:
                    collected.append(txt)
        elif page == ep:
            # Case 2b: Ending page (collect from start to el)
            for (li, txt) in lines_this_page:
                if li < el:
                    collected.append(txt)
            break
        else:
            # Case 2c: Middle pages (collect everything)
            for (li, txt) in lines_this_page:
                collected.append(txt)

        page += 1
        if page > ep:
            break

    return collected

def build_chunks_single_pass(pdf_path: str) -> list:
    """
    Main function: Build hierarchical chunks from PDF.

    Pipeline:
    1. Read all page lines
    2. Extract outline headings
    3. Locate exact line positions for headings
    4. Iterate through heading pairs, gathering text between them
    5. Split large chunks if needed

    Args:
        pdf_path: Path to PDF file

    Returns:
        List of chunk dictionaries with keys:
        - 'title': Section heading
        - 'level': Hierarchy level
        - 'content': Full section text
        - 'pdf_name': Source PDF filename

    Note: This is a "single pass" algorithm - we go through headings once,
    collecting text for each section without backtracking.
    """
    print(f"\nProcessing PDF: {os.path.basename(pdf_path)}")

    # Step 1: Read lines
    page_lines = read_pdf_lines(pdf_path)
    print(f"Read {len(page_lines)} pages")

    # Step 2: Read headings
    headings = read_outline_headings(pdf_path)
    if not headings:
        print("No headings found, treating entire doc as single chunk")
        return build_single_chunk_entire_doc(page_lines, pdf_path)

    print(f"Found {len(headings)} headings in outline")

    # Step 3: Locate line positions
    locate_heading_lines(page_lines, headings)
    print(f"Located all heading positions")

    # Step 4: Build chunks
    final_chunks = []

    # Add a doc-end marker
    if page_lines:
        maxp = max(page_lines.keys())
        last_li = page_lines[maxp][-1][0] + 9999
        doc_end = {
            'title': '_DOC_END_',
            'level': 999,
            'page_idx': maxp,
            'line_idx': last_li
        }
    else:
        doc_end = {
            'title': '_DOC_END_',
            'level': 999,
            'page_idx': 999999,
            'line_idx': 999999
        }

    extended_headings = headings + [doc_end]

    def flush_chunk(heading, lines_block):
        """Helper: Create chunk(s) from gathered lines."""
        if not lines_block:
            return

        joined = "\n".join(lines_block)

        # Split if too large
        for subc in split_by_token_limit(joined, MAX_CHUNK_WORDS):
            final_chunks.append({
                "title": heading['title'],
                "level": heading['level'],
                "content": subc,
                "pdf_name": os.path.basename(pdf_path)
            })

    # Iterate through heading pairs
    for i in range(len(extended_headings) - 1):
        h1 = extended_headings[i]
        h2 = extended_headings[i+1]
        lines_block = gather_lines_between(page_lines, h1, h2)
        flush_chunk(h1, lines_block)

    print(f"Created {len(final_chunks)} chunks")

    # TEST_MODE: Limit chunks for quick testing
    from config import TEST_MODE, TEST_MAX_DOC_CHUNKS
    if TEST_MODE and len(final_chunks) > TEST_MAX_DOC_CHUNKS:
        print(f"TEST_MODE: Limiting to first {TEST_MAX_DOC_CHUNKS} chunks (from {len(final_chunks)})")
        final_chunks = final_chunks[:TEST_MAX_DOC_CHUNKS]

    return final_chunks



## SECTION 1: LLM-BASED METADATA GENERATION

In [3]:
print(f"Initializing LLM: {OPENROUTER_MODEL}")
print("   (Connecting to OpenRouter API...)")
llm = get_llm()


# Prompt for generating code summaries
SUMMARY_PROMPT_TEMPLATE = """You are a Gurobi optimization expert.
You will receive a "code snippet" from Gurobi documentation examples.

Your task: Generate a 2-3 line natural language summary that:
- Starts with "[Code Example]" or "[Definition]" based on content type
- Mentions specific Gurobi methods used (e.g., addVar, addConstr, setObjective, optimize)
- Describes the problem domain (e.g., "transportation", "production planning")
- Highlights constraint types (e.g., "capacity constraints", "binary variables")

Important:
- Keep it to 2-3 lines maximum
- No extra commentary or headings
- Mention only methods/features that actually appear in the code
- Be specific about the optimization problem being solved

Code Snippet:
{CHUNK}

Summary:"""

# Prompt for generating keywords
KEYWORD_PROMPT_TEMPLATE = """You are a Gurobi and optimization domain expert.
You will receive a code snippet from Gurobi documentation.

Your task: Extract exactly 5-7 concise keywords or short phrases that capture:
1. Domain/scenario (e.g., "transportation", "profit maximization", "resource allocation")
2. Gurobi API features (e.g., "binary variables", "addVar", "addConstr", "setObjective")
3. Constraint types (e.g., "budget constraint", "capacity constraint")

Guidelines:
- Each keyword/phrase: 1-3 words maximum
- Focus on concepts that would appear in natural language queries
- Include both domain terms AND technical Gurobi terms
- Output format: comma-separated list
- No extra text, just the keywords

Code Snippet:
{CHUNK}

Keywords (5-7):"""


def generate_summary(llm, code_text: str) -> str:
    """
    Generate a 2-3 line natural language summary of code using LLM.

    Args:
        llm: OpenRouter LLM instance
        code_text: Code snippet text

    Returns:
        Summary string (2-3 lines)

    Example Output:
        "[Code Example] Creates a transportation model with binary variables for
        plant openings and continuous variables for shipping quantities. Uses
        addVar, addConstr, and setObjective to minimize total costs subject to
        capacity and demand constraints."
    """
    prompt = SUMMARY_PROMPT_TEMPLATE.format(CHUNK=code_text)
    response = llm.invoke(prompt).strip()
    return response


def generate_keywords(llm, code_text: str) -> str:
    """
    Extract 5-7 keywords from code using LLM.

    Args:
        llm: OpenRouter LLM instance
        code_text: Code snippet text

    Returns:
        Comma-separated keyword string

    Example Output:
        "transportation, binary variables, addVar, capacity constraints,
        minimize cost, production planning, setObjective"
    """
    prompt = KEYWORD_PROMPT_TEMPLATE.format(CHUNK=code_text)
    response = llm.invoke(prompt).strip()
    return response

Initializing LLM: google/gemini-2.5-flash
   (Connecting to OpenRouter API...)


## SECTION 2: PROCESS CODE EXAMPLES WITH METADATA

In [4]:
def process_code_chunks(chunks: list, llm) -> list:
    """
    Add metadata (summary + keywords) to each code chunk.

    This function modifies chunks in-place, adding two new keys:
    - 'summary': Natural language summary (2-3 lines)
    - 'keywords': Comma-separated keywords (5-7 items)

    Args:
        chunks: List of code chunks (from build_chunks_single_pass)
        llm: LLM instance for generation

    Returns:
        Modified chunks list with metadata

    Why This Matters:
        Raw code chunks have poor semantic similarity to natural language queries.
        By generating summaries and keywords, we create a semantic bridge that
        enables effective retrieval.

    Example:
        Original chunk: "model = gp.Model()\\nx = model.addVar(...)\\n..."
        After metadata:
            summary: "[Code Example] Creates binary decision variables..."
            keywords: "binary variables, addVar, optimization model, decision var"
    """
    print(f"\nGenerating metadata for {len(chunks)} code chunks...")

    enriched_chunks = []

    for chunk in tqdm(chunks, desc="Processing chunks"):
        # Generate metadata
        summary = generate_summary(llm, chunk['content'])
        keywords = generate_keywords(llm, chunk['content'])

        # Add metadata fields
        enriched_chunk = chunk.copy()
        enriched_chunk['summary'] = summary
        enriched_chunk['keywords'] = keywords

        enriched_chunks.append(enriched_chunk)

    return enriched_chunks

## SECTION 3: EXECUTE PIPELINE

In [5]:
print("=" * 80)
print("STAGE 2: METADATA GENERATION FOR CODE EXAMPLES")
print("=" * 80)

# Step 1: Process code examples PDF
print(f"\nProcessing code examples PDF...")
code_chunks = build_chunks_single_pass(PDF_CODE_EXAMPLES)
print(f"Extracted {len(code_chunks)} code chunks")

# TEST_MODE: Limit chunks for quick testing
from config import TEST_MODE, TEST_MAX_CODE_CHUNKS
if TEST_MODE and len(code_chunks) > TEST_MAX_CODE_CHUNKS:
    print(f"TEST_MODE: Limiting to first {TEST_MAX_CODE_CHUNKS} chunks (from {len(code_chunks)})")
    code_chunks = code_chunks[:TEST_MAX_CODE_CHUNKS]

# Step 2: Generate metadata for each chunk
enriched_chunks = process_code_chunks(code_chunks, llm)

# Step 3: Save enriched chunks
with open(CODE_CHUNKS_PATH, 'wb') as f:
    pickle.dump(enriched_chunks, f)

print(f"\nSaved {len(enriched_chunks)} enriched chunks to: {CODE_CHUNKS_PATH}")

STAGE 2: METADATA GENERATION FOR CODE EXAMPLES

Processing code examples PDF...

Processing PDF: docs-gurobi-com-examples-en-11.0.pdf
Read 721 pages
Found 310 headings in outline
Located all heading positions
Created 530 chunks
TEST_MODE: Limiting to first 20 chunks (from 530)
Extracted 20 code chunks

Generating metadata for 20 code chunks...


Processing chunks: 100%|██████████| 20/20 [00:26<00:00,  1.31s/it]


Saved 20 enriched chunks to: /Users/tasnimahmed/Downloads/tutorial/Standalone/tutorial_chunks_code.pkl


In [6]:
print(f"\n{'=' * 80}")
print(f"EXAMPLES: Code Chunks with Generated Metadata")
print(f"{'=' * 80}\n")

for i, chunk in enumerate(enriched_chunks[10:12], 1):
    print(f"{'─' * 80}")
    print(f"Code Example #{i}")
    print(f"{'─' * 80}")
    print(f"Title:    {chunk['title']}")
    print(f"PDF:      {chunk['pdf_name']}")
    print(f"Length:   {len(tokenize(chunk['content']))} words\n")

    # Show original code (truncated)
    print("Original Code (first 2000 chars):")
    print(f"{'┌' + '─' * 78 + '┐'}")
    code_preview = chunk['content'][:2000].replace('\n', '\n│ ')
    print(f"│ {code_preview}...")
    print(f"{'└' + '─' * 78 + '┘'}\n")

    # Show generated metadata
    print("Generated Summary:")
    print(f"   {chunk['summary']}\n")

    print("Generated Keywords:")
    print(f"   {chunk['keywords']}\n")


EXAMPLES: Code Chunks with Generated Metadata

────────────────────────────────────────────────────────────────────────────────
Code Example #1
────────────────────────────────────────────────────────────────────────────────
Title:    Load and solve a model from a file
PDF:      docs-gurobi-com-examples-en-11.0.pdf
Length:   313 words

Original Code (first 2000 chars):
┌──────────────────────────────────────────────────────────────────────────────┐
│ 1.1.2 Load and solve a model from a file Examples batchmode, callback, feasopt, fixanddive, lp, lpmethod, lpmod, mip2, params, sensitivity One of the most basic tasks you can perform with the Gurobi libraries is to read a model from a file, optimize it, and report the result. The lp example is a simple illustration of how this is done in the various supported Gurobi languages. While the specifics vary from one language to another, the basic structure remains the same for all languages. After initializing the Gurobi environment, the exampl